In [ ]:
try:
    from openmdao.utils.notebook_utils import notebook_mode  # noqa: F401
except ImportError:
    !python -m pip install openmdao[notebooks]

# Local Building of OpenMDAO Documentation

When developing new features for OpenMDAO, it will be necessary to build the documents locally to ensure that code embedding, formatting, and links are all working as intended. The documentation uses Jupyter notebooks executed by [papermill](https://papermill.readthedocs.io/) and rendered to HTML by [Sphinx](https://www.sphinx-doc.org/) via [MyST-NB](https://myst-nb.readthedocs.io/).

## Installation

The doc build dependencies are included in the `docs` feature of the pixi environment. From the repository root, activate the dev environment:

```bash
pixi shell -e dev
```

Alternatively, install the docs extras directly with pip:

```bash
pip install -e .[docs]
```

## Building the Docs

From the repository root, run:

```bash
python -m openmdao.devtools.build_docs build
```

This runs the full pipeline:
1. Disables SNOPT-specific cells
2. Generates the API reference (`_srcdocs/`)
3. Copies the source tree to `_executed_book/`
4. Executes all notebooks with papermill
5. Builds HTML with Sphinx

The output is at `openmdao/docs/_executed_book/_build/html/main.html`.

### Useful flags

| Flag | Description |
|------|-------------|
| `--no-exec` | Skip notebook execution; re-run Sphinx on existing `_executed_book/` output. |
| `--no-serial` | Skip serial (non-MPI) notebook execution. |
| `--no-mpi` | Skip MPI notebook execution. |
| `--fast` | Parallel Sphinx build (`-j auto`); also skips warnings-as-errors. |
| `--workers N` | Number of parallel notebook execution workers (default: cpu count). |
| `--no-rich` | Plain-text progress output (used on CI). |

### Incremental builds

Notebooks are skipped if their output in `_executed_book/` is already newer than the source. After editing a single notebook, re-running the build will only re-execute changed notebooks.

To force re-execution of all notebooks regardless of timestamps:

```bash
python -m openmdao.devtools.build_docs build --no-exec
# then manually delete the specific notebook from _executed_book/ and re-run
```

Or simply remove the entire output tree and rebuild from scratch:

```bash
python -m openmdao.devtools.build_docs clean
python -m openmdao.devtools.build_docs build
```

## Viewing the Docs

The built HTML can be opened directly as files in a browser, and most content will display correctly. However, some features — in particular, embedded N2 diagrams — are loaded via `<iframe>` and will not render when opened as `file://` URLs. This is a browser security restriction: browsers block cross-frame navigation between `file://` documents to prevent untrusted local files from accessing each other's content. The restriction applies even when both the page and the iframe source are in the same directory.

To avoid this, serve the docs over HTTP using the `view` subcommand:

```bash
python -m openmdao.devtools.build_docs view
```

This starts a local HTTP server and opens the docs in your default browser. An optional `--port` flag selects the port (default: 8000):

```bash
python -m openmdao.devtools.build_docs view --port 9000
```

Press Ctrl-C to stop the server when done.

## MPI Examples

The documentation includes several examples that demonstrate MPI features. These require MPI to be installed. MPI notebooks are identified by `"mpi": true` in the notebook's top-level metadata and are executed serially (one at a time) since each one internally spawns its own `mpiexec` subprocess.

If you don't have MPI installed, use `--no-mpi` to skip them:

```bash
python -m openmdao.devtools.build_docs build --no-mpi
```

## Style Guide

When writing new notebooks, refer to the [documentation style guide](doc_style_guide.ipynb).